## Normal & Covert Channel Attacks (message is plain text) in Chiller Controller

In [1]:
import sys
sys.path.append('C:/')
from time import sleep
import random
from BAC0.core.devices.local.object import ObjectFactory
from bacpypes.object import ScheduleObject
from bacpypes.primitivedata import Real
from BAC0 import lite, connect
from BAC0.core.devices.local.models import (
    analog_input,
    analog_output,
    binary_output,
    binary_input,
)
import csv
import ifaddr
import time

## Defined features for the controller
Analog Inputs:

- **ChilledWaterFlowrate**
- **ChilledWaterReturnTemp**
- **ChilledWaterSupplyTemp**
- **CondenserWaterFlowrate**
- **CondenserWaterReturnTemp**
- **CondenserWaterSupplyTemp**
- **CoolingTons**
- **Efficiency**
- **Power**
- **Cooling Tower7_Condenser Water Supply Temp**
- **Cooling Tower8_Condenser Water Supply Temp**
- **Cooling Tower9_Condenser Water Supply Temp**

Binary Outputs:

- **Chille dWater Pump On Off Status**
- **Condenser Water Pump On Off Status**
- **Chiller On Off Status**

In [3]:
# define device objects
def defining_objects(Chiller3):
    # start from fresh
    ObjectFactory.clear_objects()
    
    #content of the object is defined here anward
    
    Chiller3 = analog_input(
    instance=760,
    name = "ChilledWaterFlowrate",
    properties = {"units" : "cubicMetersPerHour"},
    description = "ChilledWaterFlowrate",
    presentValue = 760.0,
    )
    analog_input(
        instance=12,
        name="ChilledWaterReturnTemp",
        properties={"units" : "degreesCelsius"},
        description="ChilledWaterReturnTemp",
        presentValue=12.0,
    )
    analog_input(
        instance=7,
        name = "ChilledWaterSupplyTemp",
        properties={"units" : "degreesCelsius"},
        description="ChilledWaterSupplyTemp",
        presentValue=7.0,
    )
    analog_input(
        instance=850,
        name = "CondenserWaterFlowrate",
        properties={"units" : "cubicMetersPerHour"},
        description="CondenserWaterFlowrate",
        presentValue=850.0,
    )
    analog_input(
        instance=28,
        name = "CondenserWaterReturnTemp",
        properties={"units" : "degreesCelsius"},
        description="CondenserWaterReturnTemp",
        presentValue=28.0,
    )
    analog_input(
        instance=32,
        name = "CondenserWaterSupplyTemp",
        properties={"units" : "degreesCelsius"},
        description="CondenserWaterSupplyTemp",
        presentValue=32.0,
    )
    analog_input(
        instance=420,
        name = "CoolingTons",
        properties={"units" : "tonsRefrigeration"},
        description="CoolingTons",
        presentValue=420.0,
    )
    analog_input(
        instance=1,
        name = "Efficiency",
        properties={"units" : "percent"},
        description="Efficiency",
        presentValue=0.5,
    )
    analog_input(
        instance=1,
        name = "Power",
        properties={"units" : "kilowattHours"},
        description="Power",
        presentValue=0.5,
    )
    analog_input(
        instance=28,
        name = "CoolingTower7_CondenserWaterSupplyTemp",
        properties={"units" : "degreesCelsius"},
        description="CoolingTower7_CondenserWaterSupplyTemp",
        presentValue=28.0,
    )
    analog_input(
        instance=28,
        name = "CoolingTower8_CondenserWaterSupplyTemp",
        properties={"units" : "degreesCelsius"},
        description="CoolingTower8_CondenserWaterSupplyTemp",
        presentValue=28.0,
    )
    analog_input(
        instance=28,
        name = "CoolingTower9_CondenserWaterSupplyTemp",
        properties={"units" : "degreesCelsius"},
        description="CoolingTower9_CondenserWaterSupplyTemp",
        presentValue=28.0,
    )
    binary_output(
        instance=0,
        name = "ChilledWaterPumpOnOffStatus",
        description="ChilledWaterPumpOnOffStatus",
        presentValue=True,
        #outOfService=0,
    )
    binary_output(
        instance=0,
        name = "CondenserWaterPumpOnOffStatus",
        description="CondenserWaterPumpOnOffStatus",
        presentValue=True,
        #outOfService=0,
    )
    binary_output(
        instance=0,
        name = "ChillerOnOffStatus",
        description="ChillerOnOffStatus",
        presentValue=False,
        #outOfService=0,
    )
    return Chiller3.add_objects_to_application(Chiller)

Configuring the controller to connect to the network involves:
- **reading the network interface IP**
- **using it for controller simulation**
- **specifying the port for the back-net protocol**
- **defining an ID for the controller**

In [ ]:
adapters = ifaddr.get_adapters()
ip = None
for adapter in adapters:
    if adapter.nice_name == 'maineth':
        ip = adapter.ip_addresses[0] 
        break

AHU = connect(ip=ip, port='47808', deviceId='3330')
defining_objects(AHU)

## Convert message 

In [ ]:
def text_to_binary(message):
    # Convert text to ASCII code
    ascii_string = message.encode('ascii')
    
    # Convert ASCII code to binary code
    binary_string = bin(int.from_bytes(ascii_string, 'big'))
    
    # Remove the prefix from binary string (like '0b')
    binary_string = binary_string[2:]
    
    return binary_string

## message 

In [ ]:
message = "     Hello Bob I am Alice"

## Convert text to ASCII (0 and 1) code

In [ ]:
# Convert text to ASCII (0 and 1) code
binary_message = text_to_binary(message)
print("English message converted to ASCII (0 and 1) code:")
print(binary_message)
# Print the number of bits
num_bits = len(binary_message)
print(f"Number of bits: {num_bits}")

# Calculate and print the execution time
execution_time= num_bits*180
print(f"execution time-secend: {execution_time}")

### If the message length is incomplete, the transmission time for sensor data is 179 seconds when the ASCII character is 0, and 181 seconds when the ASCII character is 1. Once the entire message is transmitted, the transmission time for sensor data returns to normal, which is 180 seconds.

In [ ]:
def process_bit(binary_message, bit_number):
    # Number of bits in the binary string
    binary_length = len(binary_message)
    
    # If the bit number is greater than the number of bits in the binary string, return 10
    if bit_number > binary_length:
        x = 180
    # If the bit number is within the range of the binary string
    else:
        # Retrieve the value of the bit corresponding to the bit number
        bit_value = binary_message[bit_number - 1]
        
        # If the bit value is 0, set x to 9
        if bit_value == '0':
            x = 179
        # If the bit value is 1, set x to 11
        elif bit_value == '1':
            x = 181
        # If there is no corresponding bit for the given bit number, set x to 10
        else:
            x = 180
    
    return x

## Sending packets

In [5]:
start_date = time.struct_time((2012, 0, 0, 0, 0, 0, 0, 0, 0))
start_timestamp = time.mktime(start_date)

for i in range(int(start_timestamp - time.time())):
  time.sleep(1)

with open('E://') as f:
    reader = csv.reader(f)
    rows = list(reader)

row_num = 1 

while True:
    if row_num >= len(rows):
        break
        
    ChilledWaterFlowrate = float(rows[row_num][1])
    ChilledWaterReturnTemp = float(rows[row_num][3])
    ChilledWaterSupplyTemp = float(rows[row_num][4])
    CondenserWaterFlowrate = float(rows[row_num][5])
    CondenserWaterReturnTemp = float(rows[row_num][7])
    CondenserWaterSupplyTemp = float(rows[row_num][8])
    CoolingTons = float(rows[row_num][9])
    Efficiency = float(rows[row_num][10])
    Power = float(rows[row_num][11])
    CoolingTower7_CondenserWaterSupplyTemp = float(rows[row_num][13])
    CoolingTower8_CondenserWaterSupplyTemp = float(rows[row_num][14])
    CoolingTower9_CondenserWaterSupplyTemp = float(rows[row_num][15])
    ChilledWaterPumpOnOffStatus = float(rows[row_num][2])
    CondenserWaterPumpOnOffStatus = float(rows[row_num][6])
    ChillerOnOffStatus = float(rows[row_num][12])
    
    if ChilledWaterPumpOnOffStatus == 0:
      ChilledWaterPumpOnOffStatus = False
    elif ChilledWaterPumpOnOffStatus == 1:
      ChilledWaterPumpOnOffStatus = True

    if CondenserWaterPumpOnOffStatus == 0:
      CondenserWaterPumpOnOffStatus = False  
    elif CondenserWaterPumpOnOffStatus == 1:
      CondenserWaterPumpOnOffStatus = True

    if ChillerOnOffStatus == 0:
     ChillerOnOffStatus = False
    elif ChillerOnOffStatus == 1: 
     ChillerOnOffStatus = True

    Chiller["ChilledWaterFlowrate"].presentValue = ChilledWaterFlowrate 
    Chiller["ChilledWaterReturnTemp"].presentValue = ChilledWaterReturnTemp
    Chiller["ChilledWaterSupplyTemp"].presentValue = ChilledWaterSupplyTemp
    Chiller["CondenserWaterFlowrate"].presentValue = CondenserWaterFlowrate
    Chiller["CondenserWaterReturnTemp"].presentValue = CondenserWaterReturnTemp
    Chiller["CondenserWaterSupplyTemp"].presentValue = CondenserWaterSupplyTemp
    Chiller["CoolingTons"].presentValue = CoolingTons 
    Chiller["Efficiency"].presentValue = Efficiency
    Chiller["Power"].presentValue = Power
    Chiller["CoolingTower7_CondenserWaterSupplyTemp"].presentValue = CoolingTower7_CondenserWaterSupplyTemp
    Chiller["CoolingTower8_CondenserWaterSupplyTemp"].presentValue = CoolingTower8_CondenserWaterSupplyTemp
    Chiller["CoolingTower9_CondenserWaterSupplyTemp"].presentValue = CoolingTower9_CondenserWaterSupplyTemp
    Chiller["ChilledWaterPumpOnOffStatus"].presentValue = ChilledWaterPumpOnOffStatus
    Chiller["CondenserWaterPumpOnOffStatus"].presentValue = CondenserWaterPumpOnOffStatus
    Chiller["ChillerOnOffStatus"].presentValue = ChillerOnOffStatus
    
    print("Chiller ChilledWaterFlowrate :", Chiller["ChilledWaterFlowrate"].presentValue) 
    print("Chiller ChilledWaterReturnTemp :", Chiller["ChilledWaterReturnTemp"].presentValue)   
    print("Chiller ChilledWaterSupplyTemp :", Chiller["ChilledWaterSupplyTemp"].presentValue) 
    print("Chiller CondenserWaterFlowrate :", Chiller["CondenserWaterFlowrate"].presentValue) 
    print("Chiller CondenserWaterReturnTemp :", Chiller["CondenserWaterReturnTemp"].presentValue) 
    print("Chiller CondenserWaterSupplyTemp :", Chiller["CondenserWaterSupplyTemp"].presentValue) 
    print("Chiller CoolingTons :", Chiller["CoolingTons"].presentValue)   
    print("Chiller Efficiency :", Chiller["Efficiency"].presentValue) 
    print("Chiller Power :", Chiller["Power"].presentValue) 
    print("Chiller CoolingTower7_CondenserWaterSupplyTemp :", Chiller["CoolingTower7_CondenserWaterSupplyTemp"].presentValue) 
    print("Chiller CoolingTower8_CondenserWaterSupplyTemp :", Chiller["CoolingTower8_CondenserWaterSupplyTemp"].presentValue) 
    print("Chiller CoolingTower9_CondenserWaterSupplyTemp :", Chiller["CoolingTower9_CondenserWaterSupplyTemp"].presentValue)   
    print("Chiller ChilledWaterPumpOnOffStatus :", Chiller["ChilledWaterPumpOnOffStatus"].presentValue) 
    print("Chiller CondenserWaterPumpOnOffStatus :", Chiller["CondenserWaterPumpOnOffStatus"].presentValue) 
    print("Chiller ChillerOnOffStatus :", Chiller["ChillerOnOffStatus"].presentValue)
    
    
    row_num += 1
    bit_number = row_num
    x = process_bit(binary_message, bit_number)
    print("time step =", x)
    sleep(x)

print("Reached end of data")